# REST interface for DB Service

The DB Service REST interface provides direct HTTP access to DB Service APIs, making it easier to connect to a running service and perform client operations using cURL.

Use this to run queries, manage tables, and interact with DB Service over REST.

## Managing Tables
Use these calls to define and inspect table schemas in DB Service.

In [1]:
%%bash
# List tables (empty to begin with)
curl -s -X GET "http://localhost:8080/api/v0/tables" -H "Accept: application/json" | jq

[]


### Reference data and foreign keys

`instruments` is a reference table: a small, slowly changing table of instrument metadata, keyed on `sym` using `primaryKeys`. List the key column first in `columns` so that the schema matches the column order of the keyed table.

Declaring `"foreign": "instruments.sym"` on the `fxquote.sym` column links the quotes to that reference data, which lets a query read `instruments` columns using dot notation, for example `instruments.category`.

In [2]:
%%bash
# Create the 'instruments' reference table, keyed on 'sym'
curl -s -X POST "http://localhost:8080/api/v0/tables/instruments" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{
    "type": "splayed",
    "primaryKeys": ["sym"],
    "columns": [
      {"name": "sym", "type": "symbol"},
      {"name": "instrumentid", "type": "long"},
      {"name": "category", "type": "symbol"},
      {"name": "decimals", "type": "long"},
      {"name": "pipdecimals", "type": "long"}
    ]
  }' | jq

{
  "jobId": "8489eb22-9e29-9bc7-255b-4522fea4e64e",
  "status": "completed",
  "statusUri": "/api/v

0/jobs/8489eb22-9e29-9bc7-255b-4522fea4e64e",
  "startedAt": "2026-09-02T15:38:29.634902262",
  "fin

ishedAt": null,
  "table": [],
  "warnings": []
}


In [3]:
%%bash
# Create partitioned table ('fxquote'), with 'sym' as a foreign key into 'instruments'
curl -s -X POST "http://localhost:8080/api/v0/tables/fxquote" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{
    "type": "partitioned",
    "prtnCol": "ts",
    "sortColsDisk": ["sym"],
    "sortColsOrd": ["sym"],
    "columns": [
      {"name": "trddate", "type": "date"},
      {"name": "ts", "type": "timestamp"},
      {"name": "sym", "type": "symbol", "foreign": "instruments.sym", "attrMem": "grouped", "attrDisk": "parted", "attrOrd": "parted"},
      {"name": "bid", "type": "float"},
      {"name": "ask", "type": "float"}
    ]
  }' | jq

{
  "jobId": "7b76e120-a9e3-ab85-5a7b-af15a7662ea0",
  "status": "completed",
  "statusUri": "/api/v

0/jobs/7b76e120-a9e3-ab85-5a7b-af15a7662ea0",
  "startedAt": "2026-09-02T15:38:38.755689336",
  "fin

ishedAt": null,
  "table": [],
  "warnings": []
}


In [4]:
%%bash
# List tables ('fxquote' and 'instruments' tables returned)
curl -s -X GET "http://localhost:8080/api/v0/tables" -H "Accept: application/json" | jq

[
  "instruments",
  "fxquote"
]


In [5]:
%%bash
# Describe the 'fxquote' table
curl -s -X GET "http://localhost:8080/api/v0/tables/fxquote" -H "Accept: application/json" | jq

{
  "type": "partitioned",
  "prtnCol": "ts",
  "sortColsDisk": [
    "sym"
  ],
  "sortColsOrd": [


    "sym"
  ],
  "columns": [
    {
      "name": "trddate",
      "type": "date"
    },
    {
     

 "name": "ts",
      "type": "timestamp"
    },
    {
      "name": "sym",
      "type": "symbol",
 

     "foreign": "instruments.sym",
      "attrMem": "grouped",
      "attrDisk": "parted",
      "at

trOrd": "parted"
    },
    {
      "name": "bid",
      "type": "float"
    },
    {
      "name": 

"ask",
      "type": "float"
    }
  ],
  "name": "fxquote"
}


## Importing Data
DB Service supports both `file-based` and `in-memory` ingest. Any file you want to import must first be copied into the DB Service `imports` staging directory, for example: `~/.kx/db-service/data/imports/`

### Import CSV

In [6]:
%%bash
# Import a CSV file into the existing 'fxquote' table
curl -s -X POST "http://localhost:8080/api/v0/imports/files" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"table": "fxquote", "path": "fxquote.csv.gz"}' | tee /tmp/fxquote-import.json | jq

{
  "jobId": "284084ce-bc21-1a3a-5f04-29e9f46097d2",
  "database": "db",
  "jobType": "import",
  "s

tatus": "pending",
  "affectedTables": [],
  "processedPartitions": [],
  "progress": {
    "current

Partition": "",
    "partitionIndex": null,
    "partitionTotal": null,
    "currentTable": "",
    

"tableIndex": null,
    "tableTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-

09-02T15:38:55.711076222"
}


In [7]:
%%bash
# Check the status of the above import job, using the "jobId" it returned
JOB_ID=$(jq -r .jobId /tmp/fxquote-import.json)
curl -s -X GET "http://localhost:8080/api/v0/imports/$JOB_ID" -H "Accept: application/json" | jq

{
  "jobId": "284084ce-bc21-1a3a-5f04-29e9f46097d2",
  "database": "db",
  "jobType": "import",
  "s

tatus": "completed",
  "affectedTables": [
    "fxquote"
  ],
  "processedPartitions": [
    "2026-0

3-02"
  ],
  "progress": {
    "currentPartition": "",
    "partitionIndex": 1,
    "partitionTotal"

: 1,
    "currentTable": "",
    "tableIndex": 0,
    "tableTotal": 0
  },
  "error": "",
  "warning

s": [],
  "updated": "2026-09-02T15:38:56.853477537"
}


In [8]:
%%bash
# Import a parquet file into the existing 'fxquote' table
curl -s -X POST "http://localhost:8080/api/v0/imports/files" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"table": "fxquote", "path": "fxquote.parquet"}' | jq

{
  "jobId": "63772569-f281-c3f9-6ae8-4a5cd0c03e07",
  "database": "db",
  "jobType": "import",
  "s

tatus": "pending",
  "affectedTables": [],
  "processedPartitions": [],
  "progress": {
    "current

Partition": "",
    "partitionIndex": null,
    "partitionTotal": null,
    "currentTable": "",
    

"tableIndex": null,
    "tableTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-

09-02T15:39:20.373942190"
}


In [9]:
%%bash
# Import a CSV file into the 'instruments' reference table created above.
# ("createTable": "1" would create a missing table from the file, but a reference
#  table needs the primary key that only an explicit schema can declare.)
curl -s -X POST "http://localhost:8080/api/v0/imports/files" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"table": "instruments", "path": "instruments.csv"}' | jq

{
  "jobId": "4d533bf3-e639-2e08-c112-13c437b26820",
  "database": "db",
  "jobType": "import",
  "s

tatus": "pending",
  "affectedTables": [],
  "processedPartitions": [],
  "progress": {
    "current

Partition": "",
    "partitionIndex": null,
    "partitionTotal": null,
    "currentTable": "",
    

"tableIndex": null,
    "tableTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-

09-02T15:39:40.954000779"
}


### Import JSON
Users can import data directly without file staging.

In [10]:
%%bash
# Objects payload imported to 'instruments' table
curl -s -X POST "http://localhost:8080/api/v0/imports/data" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{
    "table": "instruments",
    "data": [
      {"instrumentid": 77, "sym": "USDBRL", "category": "EM", "decimals": 4, "pipdecimals": 4},
      {"instrumentid": 78, "sym": "USDKRW", "category": "EM", "decimals": 2, "pipdecimals": 2}
    ],
    "insert_as": "objects"
  }' | jq

{
  "jobId": "49718400-6d85-5898-2d9f-62fcb53021fa",
  "database": "db",
  "jobType": "import",
  "s

tatus": "pending",
  "affectedTables": [],
  "processedPartitions": [],
  "progress": {
    "current

Partition": "",
    "partitionIndex": null,
    "partitionTotal": null,
    "currentTable": "",
    

"tableIndex": null,
    "tableTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-

09-02T15:40:01.535335031"
}


In [11]:
%%bash
# Rows payload imported to the 'fxquote' table
curl -s -X POST "http://localhost:8080/api/v0/imports/data" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{
    "table": "fxquote",
    "data": [
      ["2026-01-21", "2026-01-21T10:00:00.000", "EURUSD", 901.2, 901.3],
      ["2026-01-21", "2026-01-21T10:00:00.000", "EURUSD", 901.2, 901.3]
    ],
    "columnNames": ["trddate", "ts", "sym", "bid", "ask"],
    "insert_as": "rows"
  }' | jq

# Note: for rows payload, columnNames are required.

{
  "jobId": "b5675b14-d525-f4b8-0c2d-8f272466fc11",
  "database": "db",
  "jobType": "import",
  "s

tatus": "pending",
  "affectedTables": [],
  "processedPartitions": [],
  "progress": {
    "current

Partition": "",
    "partitionIndex": null,
    "partitionTotal": null,
    "currentTable": "",
    

"tableIndex": null,
    "tableTotal": null
  },
  "error": "",
  "warnings": [],
  "updated": "2026-

09-02T15:40:22.116861718"
}


## Querying Tables
Run structured, SQL, or q queries against DB Service.

In [12]:
%%bash
# Structured query
curl -s -X POST "http://localhost:8080/api/v0/query/simple" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"table": "fxquote", "startTS": "2026.03.02D00:00:00", "endTS": "2026.03.03D00:00:00", "limit": 2}' | jq

{
  "header": {
    "corr": "a580a407-3b7e-42df-9b87-912adeabff19",
    "logCorr": "a580a407-3b7e-42

df-9b87-912adeabff19",
    "gwNode": "b3cb08dc8446",
    "version": 0,
    "rcvTS": "2026-09-02T15:4

0:42.688000000",
    "http": "json",
    "api": ".query.simple",
    "rcNode": "8370751e765b",
    "

refVintage": -9223372036854776000,
    "rc": 0,
    "ac": 0,
    "ai": "",
    "limitApplied": true,


    "aggNode": "9b12b83852c2"
  },
  "payload": [
    {
      "trddate": "2026-03-02",
      "ts": 

"2026-03-02T00:00:00.000000000",
      "sym": "AUDUSD",
      "bid": 0.67091,
      "ask": 0.67094
 

   },
    {
      "trddate": "2026-03-02",
      "ts": "2026-03-02T00:00:06.000000000",
      "sym":

 "AUDUSD",
      "bid": 0.67095,
      "ask": 0.67097
    }
  ]
}


A foreign key lets a structured query reach into reference data using `table.column` dot notation. Dot columns can be used in `agg`, `groupBy` and `filter`, and are returned under their dotted name.

In [13]:
%%bash
# Structured query joining reference data over the 'sym' foreign key:
# 'instruments.category' is returned alongside the quotes, and is filtered on 'Major'
curl -s -X POST "http://localhost:8080/api/v0/query/simple" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{
    "table": "fxquote",
    "startTS": "2026.03.02D00:00:00",
    "endTS": "2026.03.02D00:00:10",
    "agg": ["ts", "sym", "bid", "ask", "instruments.category"],
    "filter": [["=", "instruments.category", "Major"]],
    "sortCols": ["ts"]
  }' | jq

{
  "header": {
    "corr": "79d4223d-cd2c-42a5-83ab-f0b8a3eec8ee",
    "logCorr": "79d4223d-cd2c-42

a5-83ab-f0b8a3eec8ee",
    "gwNode": "b3cb08dc8446",
    "version": 0,
    "rcvTS": "2026-09-02T15:4

0:46.745000000",
    "http": "json",
    "api": ".query.simple",
    "rcNode": "8370751e765b",
    "

refVintage": -9223372036854776000,
    "rc": 0,
    "ac": 0,
    "ai": "",
    "limitApplied": false

,
    "aggNode": "9b12b83852c2"
  },
  "payload": [
    {
      "ts": "2026-03-02T00:00:00.000000000

...

     "ts": "2026-03-02T00:00:09.000000000",
      "sym": "GBPUSD",
      "bid": 1.34196,
      "ask"

: 1.342,
      "instruments.category": "Major"
    },
    {
      "ts": "2026-03-02T00:00:09.0000000

00",
      "sym": "USDCAD",
      "bid": 1.38746,
      "ask": 1.38751,
      "instruments.category"

: "Major"
    },
    {
      "ts": "2026-03-02T00:00:09.000000000",
      "sym": "USDJPY",
      "bi

d": 158.154,
      "ask": 158.158,
      "instruments.category": "Major"
    }
  ]
}


In [14]:
%%bash
# SQL query
curl -s -X POST "http://localhost:8080/api/v0/query/sql" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"query": "SELECT * FROM instruments WHERE category LIKE '\''EM'\''"}' | jq

{
  "header": {
    "corr": "6f094843-f34a-482c-bcdb-440284b5416c",
    "logCorr": "6f094843-f34a-48

2c-bcdb-440284b5416c",
    "gwNode": "b3cb08dc8446",
    "version": 0,
    "rcvTS": "2026-09-02T15:4

0:50.830000000",
    "http": "json",
    "api": ".query.sql",
    "rcNode": "8370751e765b",
    "ref

Vintage": -9223372036854776000,
    "rc": 0,
    "ac": 0,
    "ai": "",
    "aggNode": "9b12b83852c2

"
  },
  "payload": [
    {
      "sym": "CHFZAR",
      "instrumentid": 14,
      "category": "EM",

...

ipdecimals": 4
    },
    {
      "sym": "ZARJPY",
      "instrumentid": 76,
      "category": "EM",


      "decimals": 3,
      "pipdecimals": 2
    },
    {
      "sym": "USDBRL",
      "instrumentid

": 77,
      "category": "EM",
      "decimals": 4,
      "pipdecimals": 4
    },
    {
      "sym":

 "USDKRW",
      "instrumentid": 78,
      "category": "EM",
      "decimals": 2,
      "pipdecimals

": 2
    }
  ]
}


In [15]:
%%bash
# QSQL query
curl -s -X POST "http://localhost:8080/api/v0/query/q" \
  -H "Accept: application/json" \
  -H "Content-Type: application/json" \
  -d '{"query": "select o:first bid,h:max bid,l:min bid,c:last bid by trddate,sym from fxquote"}' | jq

{
  "header": {
    "corr": "6d7ff1af-0b64-44b2-b23f-e45dab919448",
    "logCorr": "6d7ff1af-0b64-44

b2-b23f-e45dab919448",
    "gwNode": "b3cb08dc8446",
    "version": 0,
    "rcvTS": "2026-09-02T15:4

0:54.894000000",
    "http": "json",
    "api": ".query.q",
    "rcNode": "8370751e765b",
    "refVi

ntage": -9223372036854776000,
    "rc": 0,
    "ac": 0,
    "ai": "",
    "aggNode": "9b12b83852c2"


  },
  "payload": [
    {
      "trddate": "2026-01-21",
      "sym": "EURUSD",
      "o": 901.2,
  

...

6-03-03",
      "sym": "GBPUSD",
      "o": 1.34401,
      "h": 1.34588,
      "l": 1.33996,
      "

c": 1.34176
    },
    {
      "trddate": "2026-03-03",
      "sym": "USDCAD",
      "o": 1.38338,
 

     "h": 1.38441,
      "l": 1.37855,
      "c": 1.3844
    },
    {
      "trddate": "2026-03-03",


      "sym": "USDJPY",
      "o": 158.177,
      "h": 158.529,
      "l": 157.746,
      "c": 158.4

61
    }
  ]
}


**Return format:** `return_as` may be `json`, `pandas`, or `pykx`. If omitted, it defaults to `json`.

## Deleting tables
A table that is the target of a foreign key cannot be dropped while the referencing table still exists, so drop `fxquote` before `instruments`.

In [16]:
%%bash
# List tables (expected: 'fxquote' and 'instruments')
curl -s -X GET "http://localhost:8080/api/v0/tables" -H "Accept: application/json" | jq

[
  "instruments",
  "fxquote"
]


In [17]:
%%bash
# Drop the 'fxquote' table (the table holding the foreign key)
curl -s -X DELETE "http://localhost:8080/api/v0/tables/fxquote" -H "Accept: application/json" | jq

{
  "jobId": "193a9b31-1309-8497-e517-1f810e182fbb",
  "status": "completed",
  "statusUri": "/api/v

0/jobs/193a9b31-1309-8497-e517-1f810e182fbb",
  "startedAt": "2026-09-02T15:41:03.218358012",
  "fin

ishedAt": null,
  "table": [],
  "warnings": []
}


In [18]:
%%bash
# Drop the 'instruments' table
curl -s -X DELETE "http://localhost:8080/api/v0/tables/instruments" -H "Accept: application/json" | jq

{
  "jobId": "0557d6b2-c6bb-27fd-c6e9-e0297265aebb",
  "status": "completed",
  "statusUri": "/api/v

0/jobs/0557d6b2-c6bb-27fd-c6e9-e0297265aebb",
  "startedAt": "2026-09-02T15:41:12.212391925",
  "fin

ishedAt": null,
  "table": [],
  "warnings": []
}


In [19]:
%%bash
# List tables (expected: both tables are gone)
curl -s -X GET "http://localhost:8080/api/v0/tables" -H "Accept: application/json" | jq

[]
